# 블록 1. 라이브러리 Import & 설정값 정의
  - RAG 구축에 필요한 도구를 불러오고, 모든 실험에서 공통으로 쓸 하이퍼파라미터를 한 곳에 모음
  - LangChain 에코시스템의 문서 로더, 텍스트 분할기, 임베딩, Chroma 벡터DB, 그리고 OpenAI 의 GPT-4o 를 사용
  - 그리고 모든 실험에서 변수를 통제하기 위해 chunk size 800, overlap 200
    (이유 : 800은 한국어 한 단락 정도를 담을 수 있어 의미 단위가 잘 보존됩니다. 더 크면 검색 정확도가 떨어지고, 더 작으면 문맥이 끊깁니다.)
  - 임베딩은 text-embedding-3-large, distance metric은 cosine, Top-k는 3으로 하나의 상수 블록에 통일
    (이유 :K 를 늘리면 Hit@K 는 올라가지만 LLM 에 들어가는 context 가 길어져 노이즈가 증가하고 비용도 늘어납니다. K=3 이 균형점이라 판단했고, 모든 실험을 동일 K=3 으로 통제했습니다.)
  - Baseline 과 Improved 를 비교할 때 '어디가 달라서 점수가 올랐다' 라고 확신 있게 말할 수 있습니다.
  - RUN_ID 는 실행할 때마다 새로운 폴더를 만들기 위한 타임스탬프이며, Chroma DB 가 Windows 에서 파일 잠금           
    문제가 자주 발생해서 분리

In [55]:
import os, hashlib
from datetime import datetime
from typing import Dict, List, Any
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_PATH = "./00_ShipLeak_Copilot_Integrated_Knowledge_Base.docx"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4o"
DISTANCE_METRIC = "cosine"
TOP_K = 3

# Windows Chroma DB 파일 잠금 문제를 피하기 위해 실행 시점마다 새 폴더 사용
BASELINE_COLLECTION_NAME = f"chroma-ShipLeak-baseline-{RUN_ID}"
BASELINE_PERSIST_DIRECTORY = f"./chroma_baseline_{RUN_ID}"

IMPROVED_COLLECTION_NAME = f"chroma-ShipLeak-improved-{RUN_ID}"
IMPROVED_PERSIST_DIRECTORY = f"./chroma_improved_{RUN_ID}"

HIGH_PRECISION_COLLECTION_NAME = f"chroma-ShipLeak-high-precision-{RUN_ID}"
HIGH_PRECISION_PERSIST_DIRECTORY = f"./chroma_high_precision_{RUN_ID}"

#ANCHOR_COLLECTION_NAME = f"chroma-ShipLeak-anchor-reference-{RUN_ID}"
#ANCHOR_PERSIST_DIRECTORY = f"./chroma_anchor_reference_{RUN_ID}"

 # 블록 2. 문서 읽기 & Chunk 분할
   - docx 파일을 읽고, 800자 단위로 200자씩 겹쳐가며 잘라서 chunk 25개를 만듬
     ("원본 데이터는 선박 배관/밸브 누설 진단 지식문서 한 개의 워드 파일입니다.)
   - Docx2txtLoader 로 읽어들인 다음, RecursiveCharacterTextSplitter 로 자름
   - chunk size 를 800자(chunking 500~800 tokens, overlap 100~200). 
   - overlap 을 200 으로 둔 건, 한 chunk 끝에 있던 문맥이 다음 chunk 시작에도 25% 정도 겹쳐서 들어가게 해서 
     문맥 손실을 줄이기 위함 입니다.
   - 결과적으로 25개의 chunk 로 이루어짐


In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
loader = Docx2txtLoader(DATA_PATH)
raw_document_list = loader.load_and_split(text_splitter=text_splitter)


# 블록 3. Embedding & LLM 초기화
  - env 파일에서 OpenAI API 키를 불러오고, 임베딩 모델과 LLM 을 초기화합니다.
  - ".env 파일에 보관된 OpenAI API 키를 load_dotenv() 로 불러옴
  - 임베딩은 OpenAI의 최신 모델인 text-embedding-3-large 를 썼는데, 3072차원 벡터로 한국어 성능이 우수하고 영문 전문용어와 한글 표현을 동시에 잘 됨
  - LLM 은 GPT-4o 이고, temperature=0 으로 두어 같은 질문에는 일관된 답변이 나오도록 했습니다. 진단 같은 전문 영역에서는 창의성보다 정확성과 재현성이 중요하기 때문


In [16]:
load_dotenv()
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# 블록 4. Topic Keywords 정의 (주요 개선)
  - 선박 누설 진단 도메인에는 8개의 핵심 주제
    (가스 누설, 시트 누설, 패킹 누설, 캐비테이션, 진동, RMS, FFT 분석, 이물질)
  - 각 주제마다 한글/영문/구어체 표현을 모두 모아 키워드 사전 으로 만듬
  - 이 사전 하나로 세 가지 작업을 동시에 처리
  - 첫째, 문서 chunk 의 주제 자동 분류, 
  - 둘째, chunk 에 붙일 Alias 텍스트의 기준, 
  - 셋째, 검색 결과가 정답인지 판정하는 Hit 평가의 기준
  - 즉, 도메인 지식을 코드 한 곳에 모아 재사용성과 일관성을 확보

In [11]:
TOPIC_KEYWORDS: Dict[str, List[str]] = {
    "Gas Leakage / Pipe Leakage": [
        "Gas Leakage", "Pipe Leakage", "Hydrogen Gas Leak", "가스 누설", "배관 누설",
        "밸브 누설", "Gas Detector", "압력 저하", "누설음", "환기", "비상정지"
    ],
    "Seat Leakage": [
        "Seat Leakage", "시트 누설", "Internal Leakage", "밸브 내부 누설", "시트 손상"
    ],
    "Packing Leakage": [
        "Packing Leakage", "패킹 누설", "Stem Leakage", "스템 누설", "Gland Packing", "패킹 마모"
    ],
    "Cavitation": [
        "Cavitation", "캐비테이션", "공동현상", "고주파 소음", "기포 붕괴"
    ],
    "Valve Vibration / Loose Fastener": [
        "Valve Vibration", "밸브 진동", "Pipe Vibration", "배관 진동", "Loose Fastener",
        "볼트 풀림", "체결 불량", "Mounting Issue", "저주파 진동"
    ],
    "RMS": [
        "RMS", "진동 RMS", "vibration level", "진동 증가", "상태 감시"
    ],
    "FFT Analysis": [
        "FFT", "FFT Analysis", "주파수 분석", "Frequency Analysis", "스펙트럼 분석"
    ],
    "Foreign Object": [
        "Foreign Object", "이물질", "밸브 내부 이물질", "막힘", "유량 저하"
    ],
}


# 블록 5. 주제 분류 함수 classify_topics
  - chunk 안에 어떤 주제 키워드가 들어있는지 자동으로 태그를 붙여주는 함수
  - (동작 원리) : chunk 본문을 소문자로 바꾼 다음, 8개 주제별 키워드 사전을 돌면서 하나라도 매칭되면 그 주제 태그를 
    붙임니다. 대소문자 구분 없이 매칭하기 위해 양쪽 다 lower() 처리했습니다.
  - 한 chunk 가 여러 주제를 다룰 수도 있기 때문에 — 예를 들어 'Seat Leakage 가 발생하면 RMS 가 증가한다' 같은 문장은 
    두 개 주제에 동시에 속합니다 — 리스트로 반환합니다.
  - 이 태그는 나중에 metadata 의 topics 필드에 저장되어, Hit@1 평가에서 '검색된 문서가 진짜 정답 주제냐?' 를 
    판정하는 근거가 됩니다."

In [12]:
def classify_topics(text: str) -> List[str]:
    matched_topics = []
    lower_text = text.lower()
    for topic, keywords in TOPIC_KEYWORDS.items():
        for keyword in keywords:
            if keyword.lower() in lower_text:
                matched_topics.append(topic)
                break  # 한 주제당 한 번만 추가
    return matched_topics

# 블록 6. Baseline 문서 생성 make_base_docs
  - Baseline 단계에서도 공정한 Hit 평가가 가능하도록 metadata 만 추가하고 본문은 손대지 않습니다.
  - Baseline은 원본 문서 그대로를 쓰지만, metadata 에 chunk_id 와 topics 태그 를 함께 붙였습니다. 
    본문(page_content)은 전혀 손대지 않습니다.
  - 이렇게 한 이유는, Baseline 도 Improved 와 동일한 Hit@1 평가 기준 으로 채점하기 위함입니다. 만약 Baseline에 
    topic 태그가 없으면 metadata 매칭이 실패해서 점수가 부당하게 낮아질 수 있고, 그러면 '개선 효과가 부풀려진다' 는 비판을 받을 수 있습니다.(Hit@1은 검색 결과 1등 문서가 정답 문서인지 보는 것)
  - 즉, 본문은 그대로 두되 평가 인프라만 동일하게 맞춰서 공정한 비교 기반을 만듬

In [22]:
def make_base_docs(raw_docs):
    base_docs = []
    for idx, doc in enumerate(raw_docs, start=1):
        topics = classify_topics(doc.page_content)
        base_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={
                    **doc.metadata,
                    "chunk_id": f"BASE-{idx}",
                    "chunk_type": "base_original_chunk",
                    "topics": " | ".join(topics),
                }
            )
        )
    return base_docs

base_docs = make_base_docs(raw_document_list)

print("\n" + "=" * 30)
print("Baseline 문서 metadata 보강 완료")
print("=" * 30)
print(f"Baseline Chunk 수: {len(base_docs)}")



Baseline 문서 metadata 보강 완료
Baseline Chunk 수: 25


# 블록 7. Chroma DB 생성 함수 create_chroma_db
  - 모든 실험이 같은 distance metric(cosine) 으로 DB를 만들도록 함수로 묶음
  - DB 생성 코드를 함수로 분리한 이유는 실험 4개가 똑같은 조건으로 만들어지도록 강제 하기 위함입니다.
     특히 collection_metadata={'hnsw:space': 'cosine'} 이 부분이 핵심인데, Chroma 의 기본 distance metric 은 L2(유클리드) 입니다. 그런데 텍스트 임베딩은 일반적으로 cosine similarity 가 적합합니다.
  - 첫 버전에서는 Baseline 은 L2, Improved 는 cosine 으로 다르게 만들어서, 점수가 올라간 것이 metric 차이 때문인지 
    진짜 개선 효과인지 구분이 안 됐었습니다. 이번 수정 버전에서는 모든 DB를 cosine으로 통일해서 그 문제를 잡았습니다.

In [14]:
def create_chroma_db(documents, collection_name, persist_directory):
    db = Chroma.from_documents(
        documents=documents,
        embedding=embedding,
        collection_name=collection_name,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": DISTANCE_METRIC},  # cosine 통일
    )
    return db


# 블록 8. Baseline DB 생성
 - 25개 원본 chunk 로 Baseline Chroma DB 를 만듭니다.
 - 25개 원본 chunk 가 각각 3072차원 벡터로 임베딩되어 Chroma 에 저장됩니다. 비교 실험의 출발점입니다.

In [25]:
baseline_db = create_chroma_db(
    documents=base_docs,
    collection_name=BASELINE_COLLECTION_NAME,
    persist_directory=BASELINE_PERSIST_DIRECTORY,
)

print("\n" + "=" * 60)
print("Baseline Chroma DB 생성 완료")
print("=" * 60)
print(f"Collection Name: {BASELINE_COLLECTION_NAME}")
print(f"Persist Directory: {BASELINE_PERSIST_DIRECTORY}")
print(f"Distance Metric: {DISTANCE_METRIC}")



Baseline Chroma DB 생성 완료
Collection Name: chroma-ShipLeak-baseline-20260621_181627
Persist Directory: ./chroma_baseline_20260621_181627
Distance Metric: cosine


# 블록 9. 평가용 Query & 기대 주제 매핑
 - 평가 query 10개 + 사람이 직접 라벨링한 정답 주제 매핑 + 일반화 검증용 unseen query 5개를 정의합니다.

In [26]:
query_list = [
    "밸브에서 가스가 새요?",
    "배관에서 누설이 발생하면 어떤 조치를 해야 하나요?",
    "Seat Leakage가 발생하면 어떤 증상이 나타나나요?",
    "Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?",
    "Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?",
    "밸브 진동이 심할 때 가능한 원인은 무엇인가요?",
    "RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?",
    "FFT 분석은 밸브 이상 진단에 왜 필요한가요?",
    "Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?",
    "Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?",
]

expected_topic_map = {
    "밸브에서 가스가 새요?": "Gas Leakage / Pipe Leakage",
    "배관에서 누설이 발생하면 어떤 조치를 해야 하나요?": "Gas Leakage / Pipe Leakage",
    "Seat Leakage가 발생하면 어떤 증상이 나타나나요?": "Seat Leakage",
    "Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?": "Packing Leakage",
    "Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?": "Cavitation",
    "밸브 진동이 심할 때 가능한 원인은 무엇인가요?": "Valve Vibration / Loose Fastener",
    "RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?": "RMS",
    "FFT 분석은 밸브 이상 진단에 왜 필요한가요?": "FFT Analysis",
    "Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?": "Valve Vibration / Loose Fastener",
    "Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?": "Foreign Object",
}

unseen_query_list = [
    "가스 감지기 알람이 발생하면 먼저 무엇을 해야 하나요?",
    "밸브 주변에서 이상한 소리가 날 때 확인할 항목은 무엇인가요?",
    "진동 데이터에서 RMS가 증가하면 어떤 이상을 의심할 수 있나요?",
    "주파수 분석으로 체결 불량과 누설음을 구분할 수 있나요?",
    "밸브 내부 이물질이 있으면 유량과 압력에 어떤 영향이 있나요?",
]

unseen_expected_topic_map = {
    "가스 감지기 알람이 발생하면 먼저 무엇을 해야 하나요?": "Gas Leakage / Pipe Leakage",
    "밸브 주변에서 이상한 소리가 날 때 확인할 항목은 무엇인가요?": "Gas Leakage / Pipe Leakage",
    "진동 데이터에서 RMS가 증가하면 어떤 이상을 의심할 수 있나요?": "RMS",
    "주파수 분석으로 체결 불량과 누설음을 구분할 수 있나요?": "FFT Analysis",
    "밸브 내부 이물질이 있으면 유량과 압력에 어떤 영향이 있나요?": "Foreign Object",
}

print("\n" + "=" * 80)
print("평가 Query 준비 완료")
print("=" * 80)
print(f"기본 평가 Query 수: {len(query_list)}")
print(f"신규 Unseen Query 수: {len(unseen_query_list)}")


평가 Query 준비 완료
기본 평가 Query 수: 10
신규 Unseen Query 수: 5


# 블록 10. Cosine Score 변환 함수 cosine_score_from_distance
  - Chroma가 돌려주는 distance를 0~1 사이의 similarity 점수(같은 정도) 로 변환
  - Chroma 가 cosine metric 으로 돌려주는 값은 distance 입니다. 즉 '0이면 똑같고, 클수록 다르다' 의미
  - Chroma에서 hnsw:space='cosine'을 사용하면 score는 보통 distance이며,
  - cosine similarity처럼 해석하려면 1 - distance로 변환합니다.

In [27]:
def cosine_score_from_distance(distance):
    try:
        score = 1.0 - float(distance)
    except Exception:
        score = 0.0
    return round(max(0.0, min(1.0, score)), 4)

# 블록 11. 문서 중복 제거 키 make_doc_key
 - Multi-Query 검색 시 같은 chunk 가 두 번 잡히지 않도록 고유 식별자를 생성
 - Multi-Query 검색을 하면 원문 query 와 확장 query 가 같은 chunk 를 중복으로 가져올 수 있습니다. 그걸 막기 위한 함수입니다.
 - 우선 metadata 에 있는 chunk_id, faq_id, anchor_id 같은 명시적 ID 를 먼저 봅니다. 그게 없으면 본문 자체를 MD5 해시로 변환해서 ID 로 씁니다. 본문이 똑같으면 해시도 똑같으니까 중복 판별이 가능합니다.
 - 이 ID 가 있어야 다음 블록의 후보군 dict 에서 'key 가 같은 문서는 가장 높은 점수만 남기기' 로직이 작동합니다.

In [28]:
def make_doc_key(doc):
    metadata = doc.metadata or {}
    for key_name in ["chunk_id", "faq_id", "anchor_id"]:
        if metadata.get(key_name):
            return str(metadata.get(key_name))
    return hashlib.md5(doc.page_content.encode("utf-8")).hexdigest()


#  블록 12. 주제 매칭 판정 doc_contains_topic (Hit 평가의 핵심)
  - 검색된 문서가 정답 주제를 다루고 있는지를 metadata 와 본문 키워드 두 단계로 판정합니다.
  - "이 함수가 Hit@1, Hit@K 평가의 심판 입니다.
  - 판정 로직은 두 단계입니다. 
  - 1단계: metadata 의 topics 또는 topic 필드에 정답 주제 이름이 있는지 확인. Baseline의 자동분류 태그나 FAQ chunk 의 명시적 주제 태그가 여기서 잡힙니다. 
  - 2단계: metadata가 비어있어도, chunk 본문에 그 주제의 키워드가 하나라도 등장하면 정답으로 인정. 예를 들어 'Gas Leakage' 라는 단어가 본문에 있으면 그 chunk는 '가스 누설' 주제를 다룬다고 판단합니다.

두 단계로 만든 이유는 너무 엄격하면 진짜 정답 chunk 도 놓치고, 너무 관대하면 아무 chunk 나 정답이 됩니다. 그 사이의 합리적 균형점을 잡은 겁니다.

In [29]:
def doc_contains_topic(doc, expected_topic):
    if not expected_topic: return False
    metadata = doc.metadata or {}
    metadata_topics = str(metadata.get("topics", "")) + " | " + str(metadata.get("topic", ""))
    if expected_topic.lower() in metadata_topics.lower():
        return True
    text = doc.page_content.lower()
    for keyword in TOPIC_KEYWORDS.get(expected_topic, []):
        if keyword.lower() in text:
            return True
    return False


# 블록 13. Query 확장 함수 get_expanded_queries 
  - 사용자 query 의 키워드를 보고 주제별 확장 query를 추가로 생성 합니다. 단, 원문을 합치지 않고 별개의 query 로 분리 합니다.
  - 여기가 첫 버전에서 가장 크게 고친 부분 입니다.
  - 처음에는 원문 query 와 확장 키워드를 한 줄로 합쳤습니다. 예를 들어 '밸브에서 가스가 새요? Gas Leakage Pipe Leakage 압력 저하 환기...' 이렇게요. 그런데 이렇게 하면 임베딩 벡터가 노이즈가 섞인 평균 벡터가 되어서, 오히려 원문의 의미가 흐려질 수 있습니다.
  - 그래서 수정 버전에서는 원문 query와 확장 query 를 각각 따로 검색하도록 분리했습니다. 이 함수는 검색용 query 리스트를 반환하고, 실제 검색은 다음 블록의 multi_query_search 가 담당합니다.
  - 또 마지막에 중복 제거 로직을 넣은 이유는, 사용자 query에 키워드가 없으면 원문 하나만 반환되어야 하기 때문입니다."

In [30]:
def get_expanded_queries(query: str) -> List[str]:
    """
    기존처럼 하나의 긴 query로 합치지 않고,
    원문 query와 확장 query를 분리하여 병렬 검색합니다.
    """
    queries = [query] # 원문은 항상 포함

    if any(word in query for word in ["가스", "누설", "새", "샘", "leak", "Leakage"]):
        queries.append("Gas Leakage Pipe Leakage Hydrogen Gas Leak 가스 누설 배관 누설 Gas Detector 압력 저하 환기 차단")

    if any(word in query for word in ["소리", "소음", "Noise", "이상음"]):
        queries.append("Abnormal Sound Noise Acoustic Emission 누설음 이상 소음 고주파 소음")

    if any(word in query for word in ["떨", "진동", "Vibration"]):
        queries.append("Valve Vibration Pipe Vibration Loose Fastener Mounting Issue 밸브 진동 배관 진동 체결 불량")

    if any(word in query for word in ["Seat", "시트"]):
        queries.append("Seat Leakage Internal Leakage 시트 누설 밸브 내부 누설 시트 손상")

    if any(word in query for word in ["Packing", "패킹", "스템", "Stem"]):
        queries.append("Packing Leakage Stem Leakage Gland Packing 패킹 누설 스템 누설 패킹 마모")

    if any(word in query for word in ["Cavitation", "캐비테이션", "공동"]):
        queries.append("Cavitation 캐비테이션 공동현상 고주파 소음 진동 밸브 손상")

    if "RMS" in query:
        queries.append("RMS vibration level 진동 RMS 진동 증가 이상 진단 상태 감시")

    if any(word in query for word in ["FFT", "주파수"]):
        queries.append("FFT Analysis Frequency Analysis 주파수 분석 진동 진단 스펙트럼 분석")

    if any(word in query for word in ["Foreign", "Object", "이물질"]):
        queries.append("Foreign Object 이물질 밸브 내부 이물질 막힘 유량 저하 압력 변화")

    # 중복 제거
    unique_queries = []
    for q in queries:
        if q not in unique_queries:
            unique_queries.append(q)

    return unique_queries

# 블록 14. Multi-Query 검색 함수 multi_query_search
  - 원문 + 확장 query 들을 각각 검색 → 중복 제거 → 점수순 정렬 → Top-K 반환의 4단계 파이프라인입니다.
  - 이 함수가 이번 개선의 심장부 입니다. 네 단계로 동작합니다.
  - 1단계 — Query 준비: use_query_expansion 이 True 면 확장된 query 리스트를, False 면 원문 하나만 사용합니다. 이 플래그 덕분에 같은 함수로 Baseline 과 Improved 를 모두 평가할 수 있습니다.
  - 2단계 — 병렬 검색: 각 query 에 대해 similarity_search_with_score 로 Top-K 를 가져옵니다. 그리고 raw distance 를 cosine_score_from_distance 로 안전한 0~1 점수로 변환합니다.
  - 3단계 — 중복 제거 with 점수 최댓값 유지: make_doc_key 로 만든 ID 를 dict 키로 써서, 같은 chunk 가 여러 query 에서 나오면 가장 높은 점수만 남깁니다. 이게 핵심인데, 원문에서 0.5점, 확장 query 에서 0.8점이 나왔다면 0.8점을 채택합니다.
  - 4단계 — 정렬 후 Top-K 반환: 최종 후보들을 점수 내림차순으로 정렬해서 상위 K개만 돌려줍니다.
  - 즉, 여러 검색 시점의 결과를 통합해서 가장 좋은 매칭을 찾는 알고리즘입니다.

In [32]:
def multi_query_search(
    database: Chroma,
    query: str,
    k: int = TOP_K,
    use_query_expansion: bool = False,
) -> List[Dict[str, Any]]:
    """
    원문 query와 확장 query를 각각 검색한 뒤,
    동일 문서 중복을 제거하고 가장 높은 score 기준으로 재정렬합니다.
    """
    search_queries = get_expanded_queries(query) if use_query_expansion else [query]
    candidates: Dict[str, Dict[str, Any]] = {}

    for search_query in search_queries:
        docs_with_distances = database.similarity_search_with_score(search_query, k=k)

        for rank, (doc, raw_distance) in enumerate(docs_with_distances, start=1):
            safe_score = cosine_score_from_distance(raw_distance)
            doc_key = make_doc_key(doc)

            if doc_key not in candidates or safe_score > candidates[doc_key]["score"]:
                candidates[doc_key] = {
                    "doc": doc,
                    "score": safe_score,
                    "raw_distance": raw_distance,
                    "search_query_used": search_query,
                    "rank_in_each_query": rank,
                }

    ranked_results = sorted(candidates.values(), key=lambda x: x["score"], reverse=True)
    return ranked_results[:k]

# 블록 15. 평가 함수 evaluate_retrieval
  - query 10개 각각에 대해 Top Score, 평균 Score, Hit@1, Hit@K 4가지 지표를 동시에 계산하고 표로 정리합니다.
  - 이 함수가 모든 실험을 통일된 방식으로 채점 합니다. 4가지 지표를 동시에 측정합니다.
  - Top Score: 1등으로 검색된 chunk 의 유사도 점수. 검색이 얼마나 강하게 매칭됐는지를 봅니다. 
    Average Score: Top-K 점수 평균. 상위 결과들의 전반적 품질입니다. Hit@1: 1등 chunk 가 정말 정답 주제냐? — 가장 중요한 실용 지표입니다. Hit@K: Top-K 안에 정답이 하나라도 있냐? — 재현율(recall) 개념.
  - 처음 버전은 Top Score 하나만 봤기 때문에, '점수는 올랐는데 엉뚱한 문서가 1등' 인 상황을 잡지 못했습니다. Hit 지표를 추가하면서 '진짜로 맞는 문서를 가져왔는지' 까지 확인할 수 있게 됐습니다.

In [33]:
def evaluate_retrieval(
    database: Chroma,
    queries: List[str],
    expected_map: Dict[str, str],
    k: int = TOP_K,
    experiment_name: str = "Experiment",
    use_query_expansion: bool = False,
) -> pd.DataFrame:
    
    """Top Score, Average Score, Hit@1, Hit@K를 함께 평가합니다."""
    rows = []

    for query in queries:
        expected_topic = expected_map.get(query, "")
        ranked_results = multi_query_search(
            database=database,
            query=query,
            k=k,
            use_query_expansion=use_query_expansion,
        )

        scores = [item["score"] for item in ranked_results]
        docs = [item["doc"] for item in ranked_results]

        top_score = scores[0] if scores else 0.0
        avg_score = round(sum(scores) / len(scores), 4) if scores else 0.0

        hit_at_1 = doc_contains_topic(docs[0], expected_topic) if docs else False
        hit_at_k = any(doc_contains_topic(doc, expected_topic) for doc in docs)

        top_doc = docs[0] if docs else None
        top_metadata = top_doc.metadata if top_doc else {}

        rows.append({
            "Experiment": experiment_name,
            "Query": query,
            "Expected Topic": expected_topic,
            "Top-k": k,
            "Top Score": top_score,
            "Average Score": avg_score,
            "Hit@1": hit_at_1,
            f"Hit@{k}": hit_at_k,
            "Top Raw Distance": ranked_results[0]["raw_distance"] if ranked_results else None,
            "Search Query Used": ranked_results[0]["search_query_used"] if ranked_results else "",
            "Top Chunk Type": top_metadata.get("chunk_type", "") if top_metadata else "",
            "Top Source": top_metadata.get("source", "") if top_metadata else "",
            "Top Topic Metadata": top_metadata.get("topics", top_metadata.get("topic", "")) if top_metadata else "",
            "Top Document Preview": top_doc.page_content[:300].replace("\n", " ") if top_doc else "",
        })

    return pd.DataFrame(rows)

# 블록 16. 결과 요약 함수 summarize_results
 - 실험별로 10개 query 결과를 평균/비율로 한 줄씩 압축한 비교표를 만듭니다.
 - "10개 query 결과를 실험별로 한 줄로 요약하는 함수입니다.
 - 점수는 평균을 내고, Hit@1 과 Hit@K 는 비율(맞춘 개수 / 전체) 로 변환합니다. 예를 들어 Hit@1 비율이 0.8 이면 '10개 중 8개를 1등에서 정확히 맞췄다' 는 뜻입니다.

In [36]:
def summarize_results(result_df: pd.DataFrame) -> pd.DataFrame:
    """실험별 평균 점수 및 Hit 비율 요약."""
    hit_k_col = f"Hit@{TOP_K}"

    summary = result_df.groupby("Experiment").agg(
        질문수=("Query", "count"),
        평균_Top_Score=("Top Score", "mean"),
        평균_Average_Score=("Average Score", "mean"),
        Hit1_비율=("Hit@1", "mean"),
        HitK_비율=(hit_k_col, "mean"),
    ).reset_index()

    for col in ["평균_Top_Score", "평균_Average_Score", "Hit1_비율", "HitK_비율"]:
        summary[col] = summary[col].round(4)

    return summary


# 블록 17. Baseline 평가 실행
 - 손대지 않은 원본 25 chunk DB 로 평가해서 비교의 기준점 을 만듭니다.
 - 이제 첫 실험인 Baseline 평가입니다. 원본 25 chunk 그대로, query expansion 도 끄고, 사용자 질문 그대로 검색합니다. 여기서 나온 점수가 모든 개선 효과를 비교하는 기준점 이 됩니다

In [38]:
baseline_result = evaluate_retrieval(
    database=baseline_db,
    queries=query_list,
    expected_map=expected_topic_map,
    k=TOP_K,
    experiment_name="Baseline: Original Chunk + Cosine",
    use_query_expansion=False,
)

print("\n" + "=" * 50)
print("Baseline 검색 성능 평가 완료")
print("=" * 50)
display(baseline_result)



Baseline 검색 성능 평가 완료


,Experiment,Query,Expected Topic,Top-k,Top Score,Average Score,Hit@1,Hit@3,Top Raw Distance,Search Query Used,Top Chunk Type,Top Source,Top Topic Metadata,Top Document Preview
0,Baseline: Original Chunk + Cosine,밸브에서 가스가 새요?,Gas Leakage / Pipe Leakage,3,0.4782,0.4782,True,True,0.521823,밸브에서 가스가 새요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage,"영문 검색 키워드 Gas Leakage, Pipe Leakage, Colorles..."
1,Baseline: Original Chunk + Cosine,배관에서 누설이 발생하면 어떤 조치를 해야 하나요?,Gas Leakage / Pipe Leakage,3,0.5244,0.5244,True,True,0.475626,배관에서 누설이 발생하면 어떤 조치를 해야 하나요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage,KB-11. Emergency Isolation / Ventilation 항목 ...
2,Baseline: Original Chunk + Cosine,Seat Leakage가 발생하면 어떤 증상이 나타나나요?,Seat Leakage,3,0.5143,0.5143,True,True,0.485679,Seat Leakage가 발생하면 어떤 증상이 나타나나요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage | Seat Leakage | RM...,대표 질문 Seat Leakage 증상은? Seat Leakage 원인은? Sea...
3,Baseline: Original Chunk + Cosine,Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?,Packing Leakage,3,0.6576,0.6576,True,True,0.342441,Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage | Packing Leakage |...,Packing Leakage Packing Leakage 조치사항은? / Stem...
4,Baseline: Original Chunk + Cosine,Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?,Cavitation,3,0.5877,0.5877,True,True,0.412326,Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage | Cavitation | FFT ...,구분 RAG 지식 내용 대표 증상 - Stem 부위 미세 누설 - Spectr...
5,Baseline: Original Chunk + Cosine,밸브 진동이 심할 때 가능한 원인은 무엇인가요?,Valve Vibration / Loose Fastener,3,0.5455,0.5455,True,True,0.454533,밸브 진동이 심할 때 가능한 원인은 무엇인가요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Valve Vibration / Loose Fastener | RMS,가능 원인 - Bearing 마모 - Actuator 기어 백래시 - 윤활 부족 ...
6,Baseline: Original Chunk + Cosine,RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?,RMS,3,0.4819,0.4819,True,True,0.518078,RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,RMS,한국어 질문과 영어 기술용어가 모두 검색되도록 한글/영문 키워드를 병기한다. AI...
7,Baseline: Original Chunk + Cosine,FFT 분석은 밸브 이상 진단에 왜 필요한가요?,FFT Analysis,3,0.5442,0.5442,True,True,0.455788,FFT 분석은 밸브 이상 진단에 왜 필요한가요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage | Seat Leakage | Pa...,< 2821.9 2821.9~3174.6 3174.6~3527.3 >= 352...
8,Baseline: Original Chunk + Cosine,Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?,Valve Vibration / Loose Fastener,3,0.6056,0.6056,True,True,0.394365,Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Valve Vibration / Loose Fastener | RMS,가능 원인 - Bearing 마모 - Actuator 기어 백래시 - 윤활 부족 ...
9,Baseline: Original Chunk + Cosine,Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?,Foreign Object,3,0.5807,0.5807,True,True,0.419321,Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?,base_original_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Cavitation | Valve Vibration / Loose Fastener ...,"영문 검색 키워드 Loose Fastener, Mounting Issue, Low..."


# 블록 18. Alias 생성 함수 make_search_alias (개선 1)
 - chunk가 다루는 주제에 해당하는 Alias만 선택적으로 추가합니다. (공통 alias 일괄 삽입 금지)
 - 개선 1번 — Topic Alias 보강 의 핵심 함수입니다.
 - 동작은 chunk 본문을 보고, 8개 주제 중 어떤 키워드가 들어있는지 확인해서 해당 주제의 Alias 텍스트만 덧붙입니다. 
   예를 들어 어떤 chunk 가 'Seat Leakage' 만 다루면 그 주제 alias 만, 'Cavitation' 과 '진동' 을 동시에 다루면 두 주제 alias 모두 추가됩니다.
 - (기존 문제점)
   첫 버전과의 가장 큰 차이점은 처음엔 '공통 키워드' 라는 이름으로 모든 chunk 에 무조건 '선박 배관, 밸브, 누설...' 같은 텍스트를 일괄로 사용. 그러니까 모든 chunk 의 임베딩 벡터가 비슷해져서 유사도가 인위적으로 부풀려졌습니다.
   
 - 이제는 해당 chunk 가 실제로 그 주제를 다룰 때만 alias 가 추가됩니다. 이게 진짜 개선입니다.

In [39]:
def make_search_alias(text: str) -> str:
    """
    주제별 Alias만 부여합니다.
    기존 코드처럼 모든 Chunk에 공통 키워드를 무조건 넣으면 Chunk 간 구분력이 낮아져서 점수가 부풀려질 수 있습니다.
    """
    aliases = []
    lower_text = text.lower()

    if any(keyword.lower() in lower_text for keyword in TOPIC_KEYWORDS["Gas Leakage / Pipe Leakage"]):
        aliases.append("""
[Gas Leakage / Pipe Leakage 검색 보강]
대표 질문:
- 밸브에서 가스가 새요?
- 배관에서 누설이 발생하면 어떤 조치를 해야 하나요?
검색 키워드:
가스 누설, 배관 누설, 밸브 누설, Gas Leakage, Pipe Leakage, Hydrogen Gas Leak,
Gas Detector, 압력 저하, 누설음, 환기, 차단, 비상정지
""")

    if any(keyword.lower() in lower_text for keyword in TOPIC_KEYWORDS["Seat Leakage"]):
        aliases.append("""
[Seat Leakage 검색 보강]
대표 질문:
- Seat Leakage가 발생하면 어떤 증상이 나타나나요?
- 밸브 시트 누설 증상은 무엇인가요?
검색 키워드:
Seat Leakage, 시트 누설, Internal Leakage, 밸브 내부 누설, 압력 저하, 누설음, RMS 증가
""")

    if any(keyword.lower() in lower_text for keyword in TOPIC_KEYWORDS["Packing Leakage"]):
        aliases.append("""
[Packing Leakage 검색 보강]
대표 질문:
- Packing 누설이 발생하면 무엇을 점검해야 하나요?
- 스템 부위에서 누설이 발생합니다.
검색 키워드:
Packing Leakage, 패킹 누설, Stem Leakage, 스템 누설, Gland Packing, 체결 상태, 씰링
""")

    if any(keyword.lower() in lower_text for keyword in TOPIC_KEYWORDS["Cavitation"]):
        aliases.append("""
[Cavitation 검색 보강]
대표 질문:
- Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?
- 캐비테이션 소음이 발생합니다.
검색 키워드:
Cavitation, 캐비테이션, 공동현상, 고주파 소음, 진동, 밸브 손상, 압력 변화
""")

    if any(keyword.lower() in lower_text for keyword in TOPIC_KEYWORDS["Valve Vibration / Loose Fastener"]):
        aliases.append("""
[Valve Vibration / Loose Fastener 검색 보강]
대표 질문:
- 밸브 진동이 심할 때 가능한 원인은 무엇인가요?
- Loose Fastener가 있으면 어떤 현상이 발생하나요?
검색 키워드:
Valve Vibration, 밸브 진동, Pipe Vibration, 배관 진동, Loose Fastener,
볼트 풀림, 체결 불량, Mounting Issue, 저주파 진동
""")

    if any(keyword.lower() in lower_text for keyword in TOPIC_KEYWORDS["RMS"]):
        aliases.append("""
[RMS 검색 보강]
대표 질문:
- RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?
- 진동 RMS가 높으면 어떤 문제가 있나요?
검색 키워드:
RMS, 진동 RMS, vibration level, 진동 증가, 이상 진단, 상태 감시
""")

    if any(keyword.lower() in lower_text for keyword in TOPIC_KEYWORDS["FFT Analysis"]):
        aliases.append("""
[FFT / Frequency Analysis 검색 보강]
대표 질문:
- FFT 분석은 밸브 이상 진단에 왜 필요한가요?
- 주파수 분석으로 무엇을 알 수 있나요?
검색 키워드:
FFT Analysis, 주파수 분석, Frequency Analysis, 진동 진단, 이상 주파수, 고주파, 저주파
""")

    if any(keyword.lower() in lower_text for keyword in TOPIC_KEYWORDS["Foreign Object"]):
        aliases.append("""
[Foreign Object 검색 보강]
대표 질문:
- Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?
- 밸브 내부 이물질 문제는 무엇인가요?
검색 키워드:
Foreign Object, 이물질, 밸브 내부 이물질, 막힘, 유량 저하, 이상 소음, 압력 변화
""")

    return "\n".join(aliases)

# 블록 19. Alias 보강 문서 생성 make_enriched_docs
  - 각 chunk 에 alias 를 앞쪽에 덧붙이고, 추가됐는지 여부를 metadata 에 기록합니다.
  - 이 함수가 실제로 25개 chunk 각각에 alias 를 입혀줍니다.
  - 두 가지 디테일이 있습니다. 
  - 첫째, alias 가 비어있는 chunk — 즉, 8개 주제에 모두 해당하지 않는 chunk — 는 원본 그대로 둡니다. 억지로 alias 를 만들지 않습니다. 
  - 둘째, alias_added라는 metadata 플래그를 추가해서, 나중에 '실제로 몇 개 chunk 에 alias 가 들어갔는지' 추적할 수 있게 했습니다.
  - 그리고 alias 는 본문 앞쪽에 붙입니다. 검색용 키워드가 문서 시작에 있어야 임베딩 가중치에 더 잘 반영되는 경향이 있기 때문입니다.

In [40]:
def make_enriched_docs(base_docs: List[Document]) -> List[Document]:
    enriched_docs = []

    for idx, doc in enumerate(base_docs, start=1):
        alias_text = make_search_alias(doc.page_content)

        if alias_text.strip():
            enriched_content = f"""
[검색 보강 Alias]
{alias_text}

[원본 문서 Chunk]
{doc.page_content}
"""
        else:
            enriched_content = f"""
[원본 문서 Chunk]
{doc.page_content}
"""

        enriched_docs.append(
            Document(
                page_content=enriched_content,
                metadata={
                    **doc.metadata,
                    "chunk_id": f"ALIAS-{idx}",
                    "chunk_type": "alias_enriched_chunk",
                    "alias_added": bool(alias_text.strip()),
                }
            )
        )

    return enriched_docs


enriched_docs = make_enriched_docs(base_docs)

print("\n" + "=" * 80)
print("Alias 보강 문서 생성 완료")
print("=" * 80)
print(f"Alias 보강 Chunk 수: {len(enriched_docs)}")
print(f"Alias가 실제 추가된 Chunk 수: {sum(1 for d in enriched_docs if d.metadata.get('alias_added'))}")

if enriched_docs:
    print("\n[Alias 보강 Chunk 미리보기]")
    print(enriched_docs[0].page_content[:1000])


Alias 보강 문서 생성 완료
Alias 보강 Chunk 수: 25
Alias가 실제 추가된 Chunk 수: 25

[Alias 보강 Chunk 미리보기]

[검색 보강 Alias]

[Gas Leakage / Pipe Leakage 검색 보강]
대표 질문:
- 밸브에서 가스가 새요?
- 배관에서 누설이 발생하면 어떤 조치를 해야 하나요?
검색 키워드:
가스 누설, 배관 누설, 밸브 누설, Gas Leakage, Pipe Leakage, Hydrogen Gas Leak,
Gas Detector, 압력 저하, 누설음, 환기, 차단, 비상정지


[Seat Leakage 검색 보강]
대표 질문:
- Seat Leakage가 발생하면 어떤 증상이 나타나나요?
- 밸브 시트 누설 증상은 무엇인가요?
검색 키워드:
Seat Leakage, 시트 누설, Internal Leakage, 밸브 내부 누설, 압력 저하, 누설음, RMS 증가


[Packing Leakage 검색 보강]
대표 질문:
- Packing 누설이 발생하면 무엇을 점검해야 하나요?
- 스템 부위에서 누설이 발생합니다.
검색 키워드:
Packing Leakage, 패킹 누설, Stem Leakage, 스템 누설, Gland Packing, 체결 상태, 씰링


[Cavitation 검색 보강]
대표 질문:
- Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?
- 캐비테이션 소음이 발생합니다.
검색 키워드:
Cavitation, 캐비테이션, 공동현상, 고주파 소음, 진동, 밸브 손상, 압력 변화


[Valve Vibration / Loose Fastener 검색 보강]
대표 질문:
- 밸브 진동이 심할 때 가능한 원인은 무엇인가요?
- Loose Fastener가 있으면 어떤 현상이 발생하나요?
검색 키워드:
Valve Vibration, 밸브 진동, Pipe Vibration, 배관 진동, Loose Fastener,
볼트 풀림, 체결 불량, Mounting Issue, 저주파 진동



# 블록 20. Improved DB 생성 & 평가
  -  Alias 보강 chunk 로 DB 를 만들고, Multi-Query 검색 까지 동시에 켜서 평가합니다.
  - "두 번째 실험 — Improved 단계입니다. 두 가지 개선이 동시에 적용됩니다. 
  - 첫째, DB 자체가 Alias 보강 chunk 로 만들어집니다. 
  - 둘째, 검색할 때 use_query_expansion=True 로 Multi-Query 병렬 검색 이 켜집니다.
  - Baseline 대비 두 변수가 동시에 바뀌긴 하지만, 다음 실험에서 expansion on/off 를 따로 비교해서 어느 쪽이 더 큰 기여를 했는지 도 분리해서 봅니다."

In [41]:
improved_db = create_chroma_db(
    documents=enriched_docs,
    collection_name=IMPROVED_COLLECTION_NAME,
    persist_directory=IMPROVED_PERSIST_DIRECTORY,
)

improved_result = evaluate_retrieval(
    database=improved_db,
    queries=query_list,
    expected_map=expected_topic_map,
    k=TOP_K,
    experiment_name="Improved: Topic Alias + Cosine + Multi Query",
    use_query_expansion=True,
)

print("\n" + "=" * 80)
print("Improved 검색 성능 평가 완료")
print("=" * 80)
display(improved_result)


Improved 검색 성능 평가 완료


,Experiment,Query,Expected Topic,Top-k,Top Score,Average Score,Hit@1,Hit@3,Top Raw Distance,Search Query Used,Top Chunk Type,Top Source,Top Topic Metadata,Top Document Preview
0,Improved: Topic Alias + Cosine + Multi Query,밸브에서 가스가 새요?,Gas Leakage / Pipe Leakage,3,0.6798,0.6445,True,True,0.320191,Gas Leakage Pipe Leakage Hydrogen Gas Leak 가스 ...,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage,[검색 보강 Alias] [Gas Leakage / Pipe Leakage 검색...
1,Improved: Topic Alias + Cosine + Multi Query,배관에서 누설이 발생하면 어떤 조치를 해야 하나요?,Gas Leakage / Pipe Leakage,3,0.6798,0.6445,True,True,0.320191,Gas Leakage Pipe Leakage Hydrogen Gas Leak 가스 ...,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage,[검색 보강 Alias] [Gas Leakage / Pipe Leakage 검색...
2,Improved: Topic Alias + Cosine + Multi Query,Seat Leakage가 발생하면 어떤 증상이 나타나나요?,Seat Leakage,3,0.6798,0.6504,False,True,0.320191,Gas Leakage Pipe Leakage Hydrogen Gas Leak 가스 ...,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage,[검색 보강 Alias] [Gas Leakage / Pipe Leakage 검색...
3,Improved: Topic Alias + Cosine + Multi Query,Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?,Packing Leakage,3,0.7012,0.6885,True,True,0.298826,Packing Leakage Stem Leakage Gland Packing 패킹 ...,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage | Packing Leakage |...,[검색 보강 Alias] [Gas Leakage / Pipe Leakage 검색...
4,Improved: Topic Alias + Cosine + Multi Query,Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?,Cavitation,3,0.6476,0.6144,True,True,0.352396,Cavitation 캐비테이션 공동현상 고주파 소음 진동 밸브 손상,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Cavitation | RMS | FFT Analysis,[검색 보강 Alias] [Cavitation 검색 보강] 대표 질문: - Ca...
5,Improved: Topic Alias + Cosine + Multi Query,밸브 진동이 심할 때 가능한 원인은 무엇인가요?,Valve Vibration / Loose Fastener,3,0.7694,0.6603,True,True,0.230622,Valve Vibration Pipe Vibration Loose Fastener ...,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Valve Vibration / Loose Fastener | RMS,[검색 보강 Alias] [Valve Vibration / Loose Faste...
6,Improved: Topic Alias + Cosine + Multi Query,RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?,RMS,3,0.7043,0.6145,True,True,0.295659,RMS vibration level 진동 RMS 진동 증가 이상 진단 상태 감시,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,RMS,[검색 보강 Alias] [RMS 검색 보강] 대표 질문: - RMS 값이 높게...
7,Improved: Topic Alias + Cosine + Multi Query,FFT 분석은 밸브 이상 진단에 왜 필요한가요?,FFT Analysis,3,0.5346,0.5299,True,True,0.465400,FFT 분석은 밸브 이상 진단에 왜 필요한가요?,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage | Seat Leakage | Pa...,[검색 보강 Alias] [Gas Leakage / Pipe Leakage 검색...
8,Improved: Topic Alias + Cosine + Multi Query,Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?,Valve Vibration / Loose Fastener,3,0.6300,0.5430,True,True,0.370018,Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Valve Vibration / Loose Fastener | RMS,[검색 보강 Alias] [Valve Vibration / Loose Faste...
9,Improved: Topic Alias + Cosine + Multi Query,Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?,Foreign Object,3,0.6402,0.5877,True,True,0.359752,Foreign Object 이물질 밸브 내부 이물질 막힘 유량 저하 압력 변화,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Foreign Object,[검색 보강 Alias] [Foreign Object 검색 보강] 대표 질문: ...


# 블록 21. FAQ 지식 항목 정의 (개선 3)
  - 자주 묻는 8개 주제에 대해 "질문 + 키워드 + 답변 근거"의 3요소 구조로 정리한 지식 카드입니다.
  - 개선 3번 — FAQ 검색 전용 Chunk 의 원천 데이터입니다.
  - 8개 주제 각각에 대해 세 가지 정보를 카드 형태로 만들었습니다. 
    questions: 사용자가 실제로 물어볼 법한 자연어 질문 패턴들 
    keywords: 그 주제의 핵심 용어 (TOPIC_KEYWORDS 사전 재사용) 
    answer: 도메인 지식 기반의 표준 답변
  - 이 구조의 장점은, 사용자 질문 → FAQ chunk의 의미적 거리가 매우 가까워진다는 점입니다. 
  - 사용자가 '가스가 새요?' 라고 물으면, 원본 문서의 표 형식 텍스트보다 'questions' 에 들어있는 '밸브에서 가스가 새요?' 와 임베딩 거리가 훨씬 가깝습니다."

In [43]:
faq_knowledge_items = [
    {
        "topic": "Gas Leakage / Pipe Leakage",
        "questions": [
            "밸브에서 가스가 새요?",
            "배관에서 누설이 발생하면 어떤 조치를 해야 하나요?",
            "가스 누설이 발생하면 어떻게 해야 하나요?",
            "Gas Leakage 발생 시 증상과 조치는 무엇인가요?",
            "Pipe Leakage 발생 시 점검 항목은 무엇인가요?",
        ],
        "keywords": TOPIC_KEYWORDS["Gas Leakage / Pipe Leakage"],
        "answer": """
Gas Leakage 또는 Pipe Leakage가 발생하면 가스 감지기 알람, 압력 저하,
누설음, 배관 주변 가스 분출 또는 가스 감지 등의 증상이 나타날 수 있다.
수소 가스는 무색·무취이므로 작업자가 직접 감지하기 어렵기 때문에 Gas Detector를 통해 확인해야 한다.
권장 조치는 안전 확보, 누설 위치 표시, 밸브 격리, 환기, 비상정지, 정비 후 재점검 순서로 수행한다.
""",
    },
    {
        "topic": "Seat Leakage",
        "questions": [
            "Seat Leakage가 발생하면 어떤 증상이 나타나나요?",
            "밸브 시트 누설 증상은 무엇인가요?",
            "밸브 내부 누설 원인은 무엇인가요?",
        ],
        "keywords": TOPIC_KEYWORDS["Seat Leakage"],
        "answer": """
Seat Leakage는 밸브 시트 부위에서 내부 누설이 발생하는 현상이다.
주요 증상은 압력 저하, 누설음, RMS 증가, 밸브 차단 성능 저하 등이다.
점검 시 밸브 시트 손상, 이물질 유입, 마모, 밀봉면 손상 여부를 확인해야 한다.
""",
    },
    {
        "topic": "Packing Leakage",
        "questions": [
            "Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?",
            "스템 부위에서 누설이 발생합니다.",
            "Packing Leakage 조치사항은 무엇인가요?",
        ],
        "keywords": TOPIC_KEYWORDS["Packing Leakage"],
        "answer": """
Packing Leakage는 주로 밸브 스템 또는 Gland Packing 부위에서 발생한다.
점검 항목은 스템 부위 누설 여부, 패킹 마모, 체결 상태, Gland 조임 상태,
씰링 손상 여부이다. 필요 시 패킹 교체 또는 재조임 후 누설 여부를 재확인한다.
""",
    },
    {
        "topic": "Cavitation",
        "questions": [
            "Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?",
            "캐비테이션 소음이 발생합니다.",
            "공동현상 발생 시 조치는 무엇인가요?",
        ],
        "keywords": TOPIC_KEYWORDS["Cavitation"],
        "answer": """
Cavitation은 유체 압력 변화로 기포가 발생하고 붕괴하면서 밸브 내부에 충격을 주는 현상이다.
주요 증상은 고주파 소음, 진동 증가, 밸브 내부 손상, 성능 저하이다.
압력 조건, 유량 조건, 밸브 개도, 손상 여부를 점검해야 한다.
""",
    },
    {
        "topic": "Valve Vibration / Loose Fastener",
        "questions": [
            "밸브 진동이 심할 때 가능한 원인은 무엇인가요?",
            "밸브가 떨려요.",
            "배관 진동이 심합니다.",
            "Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?",
        ],
        "keywords": TOPIC_KEYWORDS["Valve Vibration / Loose Fastener"],
        "answer": """
밸브 또는 배관 진동이 심한 경우 Loose Fastener, Mounting Issue, 체결 불량,
배관 지지 불량, 유동 불안정 등이 원인일 수 있다.
Loose Fastener가 있으면 저주파 진동, 소음 증가, 구조 진동, 체결부 손상 가능성이 있다.
체결 상태, 지지 구조, Mounting 상태를 점검해야 한다.
""",
    },
    {
        "topic": "RMS",
        "questions": [
            "RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?",
            "진동 RMS가 높으면 어떤 문제가 있나요?",
        ],
        "keywords": TOPIC_KEYWORDS["RMS"],
        "answer": """
RMS 값이 높다는 것은 전체 진동 에너지가 증가했다는 의미이다.
이는 체결 불량, 베어링 이상, 밸브 진동, 배관 지지 문제, 유동 이상 등과 관련될 수 있다.
조치로는 진동 원인 분석, 체결 상태 확인, FFT 분석, 운전 조건 확인, 필요 시 정비가 필요하다.
""",
    },
    {
        "topic": "FFT Analysis",
        "questions": [
            "FFT 분석은 밸브 이상 진단에 왜 필요한가요?",
            "주파수 분석으로 무엇을 알 수 있나요?",
        ],
        "keywords": TOPIC_KEYWORDS["FFT Analysis"],
        "answer": """
FFT 분석은 시간 영역의 진동 신호를 주파수 영역으로 변환하여 이상 원인을 구분하는 데 사용된다.
저주파 성분은 체결 불량이나 구조 진동과 관련될 수 있고,
고주파 성분은 누설음, 베어링 이상, Cavitation 등과 관련될 수 있다.
따라서 밸브 이상 진단에서 원인 분류와 정비 판단에 필요하다.
""",
    },
    {
        "topic": "Foreign Object",
        "questions": [
            "Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?",
            "밸브 내부 이물질 문제는 무엇인가요?",
            "이물질 때문에 유량이 줄어들 수 있나요?",
        ],
        "keywords": TOPIC_KEYWORDS["Foreign Object"],
        "answer": """
Foreign Object가 밸브 내부에 있으면 유로 막힘, 유량 저하, 압력 변화,
이상 소음, 밸브 시트 손상, 누설 등이 발생할 수 있다.
점검 시 밸브 내부 이물질 유입 여부, 시트 손상, 유량 변화, 압력 변화를 확인해야 한다.
""",
    },
]

# 블록 22. FAQ Chunk 변환 make_faq_docs
 -  FAQ 지식 항목을 LangChain Document 형식 으로 변환해서 DB 에 넣을 수 있게 만듭니다.
 - 본문 텍스트는 일관된 템플릿 — '주제 / 대표 질문 / 검색 키워드 / 답변 근거' — 로 구성했고, metadata 에 faq_id 와 chunk_type='faq_high_precision_chunk' 태그를 붙여서 나중에 어떤 chunk 가 검색되었는지 추적 가능 하게 만들었습니다.
 - 8개 주제 × 1개 chunk = FAQ chunk 8개 가 생성됩니다.

In [45]:
def make_faq_docs(items: List[Dict[str, Any]]) -> List[Document]:
    faq_docs = []

    for idx, item in enumerate(items, start=1):
        content = f"""
[FAQ 검색 전용 Chunk]

주제:
{item["topic"]}

대표 질문:
{chr(10).join("- " + q for q in item["questions"])}

검색 키워드:
{", ".join(item["keywords"])}

답변 근거:
{item["answer"]}
"""

        faq_docs.append(
            Document(
                page_content=content,
                metadata={
                    "source": "manual_faq_knowledge",
                    "faq_id": f"FAQ-{idx}",
                    "topic": item["topic"],
                    "topics": item["topic"],
                    "chunk_type": "faq_high_precision_chunk",
                }
            )
        )

    return faq_docs

faq_docs = make_faq_docs(faq_knowledge_items)

print("\n" + "=" * 80)
print("FAQ 검색 전용 Chunk 생성 완료")
print("=" * 80)
print(f"FAQ Chunk 수: {len(faq_docs)}")

if faq_docs:
    print("\n[FAQ Chunk 미리보기]")
    print(faq_docs[0].page_content[:1000])


FAQ 검색 전용 Chunk 생성 완료
FAQ Chunk 수: 8

[FAQ Chunk 미리보기]

[FAQ 검색 전용 Chunk]

주제:
Gas Leakage / Pipe Leakage

대표 질문:
- 밸브에서 가스가 새요?
- 배관에서 누설이 발생하면 어떤 조치를 해야 하나요?
- 가스 누설이 발생하면 어떻게 해야 하나요?
- Gas Leakage 발생 시 증상과 조치는 무엇인가요?
- Pipe Leakage 발생 시 점검 항목은 무엇인가요?

검색 키워드:
Gas Leakage, Pipe Leakage, Hydrogen Gas Leak, 가스 누설, 배관 누설, 밸브 누설, Gas Detector, 압력 저하, 누설음, 환기, 비상정지

답변 근거:

Gas Leakage 또는 Pipe Leakage가 발생하면 가스 감지기 알람, 압력 저하,
누설음, 배관 주변 가스 분출 또는 가스 감지 등의 증상이 나타날 수 있다.
수소 가스는 무색·무취이므로 작업자가 직접 감지하기 어렵기 때문에 Gas Detector를 통해 확인해야 한다.
권장 조치는 안전 확보, 누설 위치 표시, 밸브 격리, 환기, 비상정지, 정비 후 재점검 순서로 수행한다.




# 블록 23. High Precision DB 생성 & 평가
 - Alias 보강(25개) + FAQ chunk(8개) 를 합쳐 총 33 chunk 의 강화 DB 를 만들고, Multi-Query 효과를 분리 측정 합니다.
 - "개선 3번이 적용된 단계 입니다. Alias 보강 chunk 25개 + FAQ chunk 8개 = 총 33개로 DB 를 만듭니다.
 - 그리고 같은 DB 를 두 번 평가 합니다. 한 번은 Multi-Query 켜고, 한 번은 끄고요. 이렇게 비교하면 'Multi-Query 자체의 기여도' 와 'FAQ chunk 자체의 기여도' 를 분리해서 측정할 수 있습니다.
 - 결과를 미리 말씀드리면, FAQ chunk 만 추가해도 Top Score 가 0.664 까지 올라가고, Multi-Query 까지 합치면 0.771 까지 갑니다. 두 개선이 상호 보완적으로 작동한다는 증거입니다."

In [46]:
high_precision_docs = enriched_docs + faq_docs

high_precision_db = create_chroma_db(
    documents=high_precision_docs,
    collection_name=HIGH_PRECISION_COLLECTION_NAME,
    persist_directory=HIGH_PRECISION_PERSIST_DIRECTORY,
)

high_precision_result = evaluate_retrieval(
    database=high_precision_db,
    queries=query_list,
    expected_map=expected_topic_map,
    k=TOP_K,
    experiment_name="High Precision: Alias + FAQ + Cosine + Multi Query",
    use_query_expansion=True,
)

high_precision_no_expansion_result = evaluate_retrieval(
    database=high_precision_db,
    queries=query_list,
    expected_map=expected_topic_map,
    k=TOP_K,
    experiment_name="High Precision: Alias + FAQ + Cosine",
    use_query_expansion=False,
)

print("\n" + "=" * 80)
print("High Precision 검색 성능 평가 완료")
print("=" * 80)
display(high_precision_result)


High Precision 검색 성능 평가 완료


,Experiment,Query,Expected Topic,Top-k,Top Score,Average Score,Hit@1,Hit@3,Top Raw Distance,Search Query Used,Top Chunk Type,Top Source,Top Topic Metadata,Top Document Preview
0,High Precision: Alias + FAQ + Cosine + Multi Q...,밸브에서 가스가 새요?,Gas Leakage / Pipe Leakage,3,0.6916,0.6689,True,True,0.308450,Gas Leakage Pipe Leakage Hydrogen Gas Leak 가스 ...,faq_high_precision_chunk,manual_faq_knowledge,Gas Leakage / Pipe Leakage,[FAQ 검색 전용 Chunk] 주제: Gas Leakage / Pipe Lea...
1,High Precision: Alias + FAQ + Cosine + Multi Q...,배관에서 누설이 발생하면 어떤 조치를 해야 하나요?,Gas Leakage / Pipe Leakage,3,0.6915,0.6689,True,True,0.308460,Gas Leakage Pipe Leakage Hydrogen Gas Leak 가스 ...,faq_high_precision_chunk,manual_faq_knowledge,Gas Leakage / Pipe Leakage,[FAQ 검색 전용 Chunk] 주제: Gas Leakage / Pipe Lea...
2,High Precision: Alias + FAQ + Cosine + Multi Q...,Seat Leakage가 발생하면 어떤 증상이 나타나나요?,Seat Leakage,3,0.7465,0.7062,True,True,0.253464,Seat Leakage Internal Leakage 시트 누설 밸브 내부 누설 시...,faq_high_precision_chunk,manual_faq_knowledge,Seat Leakage,[FAQ 검색 전용 Chunk] 주제: Seat Leakage 대표 질문: -...
3,High Precision: Alias + FAQ + Cosine + Multi Q...,Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?,Packing Leakage,3,0.7483,0.7137,True,True,0.251687,Packing 누설이 발생했을 때 점검해야 할 항목은 무엇인가요?,faq_high_precision_chunk,manual_faq_knowledge,Packing Leakage,[FAQ 검색 전용 Chunk] 주제: Packing Leakage 대표 질문...
4,High Precision: Alias + FAQ + Cosine + Multi Q...,Cavitation이 발생하면 밸브에 어떤 문제가 생기나요?,Cavitation,3,0.7585,0.6821,True,True,0.241536,Cavitation 캐비테이션 공동현상 고주파 소음 진동 밸브 손상,faq_high_precision_chunk,manual_faq_knowledge,Cavitation,[FAQ 검색 전용 Chunk] 주제: Cavitation 대표 질문: - C...
5,High Precision: Alias + FAQ + Cosine + Multi Q...,밸브 진동이 심할 때 가능한 원인은 무엇인가요?,Valve Vibration / Loose Fastener,3,0.7947,0.7246,True,True,0.205330,Valve Vibration Pipe Vibration Loose Fastener ...,faq_high_precision_chunk,manual_faq_knowledge,Valve Vibration / Loose Fastener,[FAQ 검색 전용 Chunk] 주제: Valve Vibration / Loos...
6,High Precision: Alias + FAQ + Cosine + Multi Q...,RMS 값이 높게 측정되면 어떤 조치를 해야 하나요?,RMS,3,0.7105,0.6732,True,True,0.289473,RMS vibration level 진동 RMS 진동 증가 이상 진단 상태 감시,faq_high_precision_chunk,manual_faq_knowledge,RMS,[FAQ 검색 전용 Chunk] 주제: RMS 대표 질문: - RMS 값이 높...
7,High Precision: Alias + FAQ + Cosine + Multi Q...,FFT 분석은 밸브 이상 진단에 왜 필요한가요?,FFT Analysis,3,0.7979,0.6227,True,True,0.202104,FFT 분석은 밸브 이상 진단에 왜 필요한가요?,faq_high_precision_chunk,manual_faq_knowledge,FFT Analysis,[FAQ 검색 전용 Chunk] 주제: FFT Analysis 대표 질문: -...
8,High Precision: Alias + FAQ + Cosine + Multi Q...,Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?,Valve Vibration / Loose Fastener,3,0.6696,0.6038,True,True,0.330392,Loose Fastener가 있으면 밸브 운전 중 어떤 현상이 나타나나요?,faq_high_precision_chunk,manual_faq_knowledge,Valve Vibration / Loose Fastener,[FAQ 검색 전용 Chunk] 주제: Valve Vibration / Loos...
9,High Precision: Alias + FAQ + Cosine + Multi Q...,Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?,Foreign Object,3,0.7600,0.6722,True,True,0.240014,Foreign Object가 밸브 내부에 있으면 어떤 문제가 발생하나요?,faq_high_precision_chunk,manual_faq_knowledge,Foreign Object,[FAQ 검색 전용 Chunk] 주제: Foreign Object 대표 질문:...


In [66]:
comparison_df = pd.concat(
    [
        baseline_result,
        improved_result,
        high_precision_no_expansion_result,
        high_precision_result,
    ],
    ignore_index=True,
)

experiment_order = [
    "Baseline: Original Chunk + Cosine",
    "Improved: Topic Alias + Cosine + Multi Query",
    "High Precision: Alias + FAQ + Cosine",
    "High Precision: Alias + FAQ + Cosine + Multi Query",
]

comparison_df["Experiment"] = pd.Categorical(
    comparison_df["Experiment"],
    categories=experiment_order,
    ordered=True,
)


def summarize_results(result_df):
    hit_k_col = f"Hit@{TOP_K}"
    summary = result_df.groupby("Experiment", observed=True, sort=False).agg(
        질문수=("Query", "count"),
        평균_Top_Score=("Top Score", "mean"),
        평균_Average_Score=("Average Score", "mean"),
        Hit1_비율=("Hit@1", "mean"),
        HitK_비율=(hit_k_col, "mean"),
    ).reset_index()

    for col in ["평균_Top_Score", "평균_Average_Score", "Hit1_비율", "HitK_비율"]:
        summary[col] = summary[col].round(4)

    return summary


summary_df = summarize_results(comparison_df)

print("\n" + "=" * 80)
print("전체 실험 요약 (실험 순서대로 정렬)")
print("=" * 80)
display(summary_df)



전체 실험 요약 (실험 순서대로 정렬)


,Experiment,질문수,평균_Top_Score,평균_Average_Score,Hit1_비율,HitK_비율
0,Baseline: Original Chunk + Cosine,10,0.5520,0.5520,1.0,1.0
1,Improved: Topic Alias + Cosine + Multi Query,10,0.6667,0.6178,0.9,1.0
2,High Precision: Alias + FAQ + Cosine,10,0.6643,0.5935,1.0,1.0
3,High Precision: Alias + FAQ + Cosine + Multi Q...,10,0.7369,0.6736,1.0,1.0


#  블록 24. Negative Query 테스트 (부풀림 검출)
  - 완전히 무관한 질문 으로 검색해서, DB 가 부적절하게 점수를 부풀리지 않는지 확인합니다.
  - Negative Query 테스트는 일종의 안전장치입니다.
  - 우리 도메인과 전혀 상관없는 질문 — 식당 메뉴, 축구, 세금 — 을 일부러 던져봅니다. 만약 잘 설계된 RAG 라면 이 질문들에 대해서는 Top Score 가 낮게 나와야 합니다. 'DB 에 답이 없으니 점수도 낮다' 가 정상입니다.
  - 그런데 만약 Alias 나 FAQ 에 너무 일반적인 키워드가 들어가 있어서 무관한 질문에도 0.5 이상의 점수가 찍힌다면, 그건 점수 부풀림 증상 입니다. 첫 버전에서 '공통 alias' 를 모든 chunk 에 박았을 때 정확히 이 문제가 있었습니다.
  - 이번엔 negative query 점수가 의미 있게 낮게 나오는지 확인해서, '개선이 진짜 의미 있는 매칭 향상' 임을 증명 합니다."

In [53]:
negative_query_list = [
    "선박 식당 메뉴는 어떻게 정하나요?",
    "축구 경기 결과를 알려주세요.",
    "법인세 신고 방법은 무엇인가요?",
]

negative_expected_map = {q: "" for q in negative_query_list}

negative_result = evaluate_retrieval(
    database=high_precision_db,
    queries=negative_query_list,
    expected_map=negative_expected_map,
    k=TOP_K,
    experiment_name="Negative Query Test: High Precision DB",
    use_query_expansion=True,
)

print("\n" + "=" * 50)
print("Negative Query 테스트")
print("=" * 50)
display(negative_result)

print("\n[Negative Query 판단 기준]")
print("- 관련 없는 질문의 Top Score가 높게 나오면 Alias/FAQ가 과도하게 일반화된 것일 수 있습니다.")
print("- 일반적으로 해당 프로젝트에서는 Negative Query의 Top Score가 주요 질문보다 낮아야 합니다.")


Negative Query 테스트


,Experiment,Query,Expected Topic,Top-k,Top Score,Average Score,Hit@1,Hit@3,Top Raw Distance,Search Query Used,Top Chunk Type,Top Source,Top Topic Metadata,Top Document Preview
0,Negative Query Test: High Precision DB,선박 식당 메뉴는 어떻게 정하나요?,,3,0.2137,0.2074,False,False,0.786262,선박 식당 메뉴는 어떻게 정하나요?,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Seat Leakage | Packing Leakage | Cavitation | ...,[검색 보강 Alias] [Seat Leakage 검색 보강] 대표 질문: - ...
1,Negative Query Test: High Precision DB,축구 경기 결과를 알려주세요.,,3,0.1769,0.1721,False,False,0.823109,축구 경기 결과를 알려주세요.,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Cavitation | RMS | FFT Analysis,[검색 보강 Alias] [Cavitation 검색 보강] 대표 질문: - Ca...
2,Negative Query Test: High Precision DB,법인세 신고 방법은 무엇인가요?,,3,0.2824,0.2761,False,False,0.717622,법인세 신고 방법은 무엇인가요?,alias_enriched_chunk,./00_ShipLeak_Copilot_Integrated_Knowledge_Bas...,Gas Leakage / Pipe Leakage | Seat Leakage | RM...,[검색 보강 Alias] [Gas Leakage / Pipe Leakage 검색...



[Negative Query 판단 기준]
- 관련 없는 질문의 Top Score가 높게 나오면 Alias/FAQ가 과도하게 일반화된 것일 수 있습니다.
- 일반적으로 해당 프로젝트에서는 Negative Query의 Top Score가 주요 질문보다 낮아야 합니다.


# 블록 25. Prompt 정의 (Baseline vs Improved)
 - 개선 4번 — Prompt Engineering. 환각 방지 지시 + 5단계 구조화 답변 형식을 추가했습니다.
 - 개선 4번 — Prompt 엔지니어링 입니다. Baseline 과 Improved 의 차이가 두 가지입니다.
 - 첫째, 환각(hallucination) 방지 지시: 'Context 에 있는 내용만 근거로 사용해라, 없으면 모른다고 답해라' 라고 명시했습니다. 안전이 중요한 선박 진단 도메인에서 LLM 이 추측으로 답변하면 큰 사고로 이어질 수 있습니다.
 - 둘째, 답변 형식 구조화: 진단 요약 / 가능한 원인 / 점검 방법 / 권장 조치 / 근거 문서 요약 — 5단계로 답을 강제합니다. 작업자가 현장에서 빠르게 행동에 옮길 수 있도록 표준 진단 리포트 형식 을 따랐습니다.
 - 이 두 가지로 답변 품질, 일관성, 신뢰성이 크게 올라갑니다.

In [56]:
TOP_K = 3
RETRIEVER_SEARCH_TYPE = "similarity"

baseline_prompt = PromptTemplate.from_template(
    """
당신은 선박 배관 및 밸브 누설 진단 전문가입니다.
아래 Context를 참고하여 질문에 답변하세요.

[Context]
{context}

[Question]
{question}

[Answer]
"""
)

improved_prompt = PromptTemplate.from_template(
    """
당신은 선박 배관 및 밸브 누설 진단 전문가입니다.

아래 [Context]에 있는 내용만 근거로 사용하여 답변하세요.
문서에 없는 내용은 추측하지 말고 "제공된 문서만으로는 확인하기 어렵습니다."라고 답변하세요.

답변 형식:
1. 진단 요약
2. 가능한 원인
3. 점검 방법
4. 권장 조치
5. 근거 문서 요약

[Context]
{context}

[Question]
{question}

[Answer]
"""
)

baseline_retriever = baseline_db.as_retriever(
    search_type=RETRIEVER_SEARCH_TYPE,
    search_kwargs={"k": TOP_K},
)

high_precision_retriever = high_precision_db.as_retriever(
    search_type=RETRIEVER_SEARCH_TYPE,
    search_kwargs={"k": TOP_K},
)

baseline_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=baseline_retriever,
    chain_type_kwargs={"prompt": baseline_prompt},
    return_source_documents=True,
)

high_precision_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=high_precision_retriever,
    chain_type_kwargs={"prompt": improved_prompt},
    return_source_documents=True,
)

print("\n" + "=" * 80)
print("QA Chain 생성 완료")
print("=" * 80)
print("baseline_qa_chain")
print("high_precision_qa_chain")


QA Chain 생성 완료
baseline_qa_chain
high_precision_qa_chain


# 블록 26. Manual RAG 함수 answer_with_manual_rag
 - Multi-Query 검색 결과를 LCEL 체인으로 LLM 에 전달해서 최종 답변까지 생성하는 통합 함수입니다.
 - 검색과 답변 생성을 하나로 잇는 함수입니다.
 - LangChain의 기본 RetrievalQA 체인은 우리가 만든 Multi-Query 병렬 검색을 직접 반영하기가 어려워서, 검색은 multi_query_search 로 직접 하고, 그 결과를 context 텍스트로 가공한 뒤에 LCEL 문법(prompt | llm | parser) 으로 LLM 에 전달했습니다.
 (RetrievalQA 는 내부 구조가 "Retriever 객체 1개 → 한 번의 검색 → LLM" 으로 고정되어 있어서, "여러 query 로 검색 → 결과 병합 → 점수순 재정렬" 같은 다단계 로직을 끼워넣을 자리가 없습니다.)
 - context 안에 검색순위, 점수, chunk type 까지 같이 넣은 이유는, 나중에 답변을 추적할 때 '어느 chunk 가 어떤 점수로 들어왔는지' 한눈에 보기 위함입니다. 

In [57]:
def answer_with_manual_rag(
    database: Chroma,
    question: str,
    prompt: PromptTemplate = improved_prompt,
    k: int = TOP_K,
    use_query_expansion: bool = True,
) -> Dict[str, Any]:
    ranked_results = multi_query_search(
        database=database,
        query=question,
        k=k,
        use_query_expansion=use_query_expansion,
    )

    context = "\n\n".join(
        f"[검색순위 {idx}]\n"
        f"Score: {item['score']}\n"
        f"Chunk Type: {item['doc'].metadata.get('chunk_type', '')}\n"
        f"Topic: {item['doc'].metadata.get('topics', item['doc'].metadata.get('topic', ''))}\n"
        f"Content:\n{item['doc'].page_content}"
        for idx, item in enumerate(ranked_results, start=1)
    )

    rag_chain = prompt | llm | StrOutputParser()
    answer = rag_chain.invoke({"context": context, "question": question})

    return {
        "question": question,
        "answer": answer,
        "retrieved_results": ranked_results,
        "context": context,
    }

# 블록 27. 답변 예시 출력
 - 실제로 한 개 질문에 대해 검색된 chunk + LLM 답변 을 모두 출력해서 시연합니다.
 - "마지막으로 실제 작동 예시를 보여줍니다. '밸브에서 가스가 새요?' 라는 구어체 질문에 대해, High Precision DB 가 어떤 chunk 를 어떤 점수로 가져왔고, GPT-4o 가 그 근거로 어떤 5단계 구조 답변을 생성했는지 한꺼번에 볼 수 있습니다. 

In [61]:
sample_question = "밸브에서 가스가 새요?"
manual_result = answer_with_manual_rag(
    database=high_precision_db, question=sample_question,
    prompt=improved_prompt, k=TOP_K, use_query_expansion=True,
)
print(manual_result["answer"])
for idx, item in enumerate(manual_result["retrieved_results"], start=1):
    print("-" * 80)
    print(f"순위: {idx}")
    print(f"Score: {item['score']}")
    print(f"Raw Distance: {item['raw_distance']}")
    print(f"Search Query Used: {item['search_query_used']}")
    doc = item["doc"]
    print(f"Chunk Type: {doc.metadata.get('chunk_type', '')}")
    print(f"Topic: {doc.metadata.get('topics', doc.metadata.get('topic', ''))}")
    print(doc.page_content[:500])


1. 진단 요약
밸브에서 가스가 새는 경우, 가스 감지기 알람, 압력 저하, 누설음, 배관 주변 가스 분출 등의 증상이 나타날 수 있습니다.

2. 가능한 원인
- 밸브, 배관, 플랜지, 가스켓, 피팅, 용접부의 누설

3. 점검 방법
- 작업 전 배관을 격리하고 압력을 제거합니다.
- 플랜지 외관, 볼트 체결 상태, 가스켓 돌출/손상 여부를 확인합니다.
- 비눗물 테스트 또는 Leak Test로 누설 위치를 확인합니다.
- 압력 유지 시험으로 누설 여부를 재검증합니다.

4. 권장 조치
- 안전 확보 후 누설 위치를 표시하고 밸브를 격리합니다.
- 환기 및 비상정지를 수행합니다.
- 볼트를 균등하게 재체결하고, 가스켓 손상 시 교체합니다.
- 플랜지 면 손상 시 보수 또는 교체합니다.
- 재조립 후 Leak Test와 Pressure Test를 수행합니다.

5. 근거 문서 요약
가스 누설이 발생하면 가스 감지기 알람, 압력 저하, 누설음 등의 증상이 나타날 수 있으며, 수소 가스는 무색·무취이므로 Gas Detector를 통해 확인해야 합니다. 권장 조치는 안전 확보, 누설 위치 표시, 밸브 격리, 환기, 비상정지, 정비 후 재점검 순서로 수행합니다. 점검 방법으로는 배관 격리, 압력 제거, 비눗물 테스트, 압력 유지 시험 등이 있으며, 권장 조치로는 볼트 재체결, 가스켓 교체, 플랜지 보수 등이 포함됩니다.
--------------------------------------------------------------------------------
순위: 1
Score: 0.6915
Raw Distance: 0.3084598779678345
Search Query Used: Gas Leakage Pipe Leakage Hydrogen Gas Leak 가스 누설 배관 누설 Gas Detector 압력 저하 환기 차단
Chunk Type: faq_high_precision_chunk
Topic: Gas Leakage / Pipe Leakage

[FAQ 검

# 결과저장

In [68]:
comparison_csv_path = f"retrieval_comparison_{RUN_ID}.csv"
summary_csv_path = f"retrieval_summary_{RUN_ID}.csv"
unseen_csv_path = f"unseen_query_result_{RUN_ID}.csv"
negative_csv_path = f"negative_query_result_{RUN_ID}.csv"

comparison_df.to_csv(comparison_csv_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")
negative_result.to_csv(negative_csv_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("결과 파일 저장 완료")
print("=" * 80)
print(comparison_csv_path)
print(summary_csv_path)
print(unseen_csv_path)
print(negative_csv_path)


결과 파일 저장 완료
retrieval_comparison_20260621_212633.csv
retrieval_summary_20260621_212633.csv
unseen_query_result_20260621_212633.csv
negative_query_result_20260621_212633.csv
